# Fitting

Arbeitsnotebook zum Fitten deiner interpolierten Spektren, angelehnt an `PEI20_3x-cole-fit_all-results (1).ipynb`.

Ablauf: Daten laden -> Zeit auf Atmosphaerenwechsel transformieren -> Spektren interpolieren -> Ableitung von epsilon real bilden -> Cole-Cole-Fit fuer einzelne Zeitpunkte und Zeitverlaeufe.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.rcParams.update({
    "figure.figsize": (9, 6),
    "axes.grid": True,
})


## Auswahl

In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "data" / "Daten final").exists():
            return path
    raise FileNotFoundError("Projektordner mit data/Daten final wurde nicht gefunden.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "Daten final"

# Messreihe fuer die ersten Fits
MATERIAL = "PEI5mgmL"
TEMPERATURE = "50°C"
MODE = "Abs"

# Zeitpunkt relativ zum Atmosphaerenwechsel fuer den Beispiel-Fit
FIT_TIME_S = 0

# Die letzten Frequenzpunkte koennen ignoriert werden, falls sie fehlerhaft sind.
IGNORE_LAST_FREQ_POINTS = 3

# Beim Fit kann neben der Ableitung von epsilon real auch epsilon imag mitgefittet werden.
FIT_EPS_IMAG = True

TIME_COL = "Time_Relative_s"
SELECTED_FILE = f"{MATERIAL}_{TEMPERATURE}_{MODE}.TXT"
DATA_DIR / SELECTED_FILE


## Daten laden und Zeit transformieren

In [ ]:
COLUMNS = [
    "Freq_Hz",
    "Time_s",
    "Eps_real",
    "Eps_imag",
    "Temp_K",
    "MTime_s",
    "Phi_deg",
    "Z_real_Ohm",
    "Z_imag_Ohm",
    "TanPhi",
]

manual_switch_points = {
    ("PEI2mgmL", "50°C", "Abs"): 132,
    ("PEI2mgmL", "50°C", "Des"): 145,
    ("PEI2mgmL", "90°C", "Abs"): 151,
    ("PEI2mgmL", "90°C", "Des"): 152,
    ("PEI5mgmL", "50°C", "Abs"): 127,
    ("PEI5mgmL", "50°C", "Des"): 145,
    ("PEI5mgmL", "60°C", "Abs"): 127,
    ("PEI5mgmL", "60°C", "Des"): 145,
    ("PEI5mgmL", "70°C", "Abs"): 127,
    ("PEI5mgmL", "70°C", "Des"): 145,
    ("PEI5mgmL", "80°C", "Abs"): 128,
    ("PEI5mgmL", "80°C", "Des"): 128,
    ("PEI5mgmL", "90°C", "Abs"): 128,
    ("PEI5mgmL", "90°C", "Des"): 128,
}


def load_measurements(data_dir=DATA_DIR, selected_file=None):
    if selected_file is None:
        file_paths = sorted(list(data_dir.glob("*.TXT")) + list(data_dir.glob("*.txt")))
    else:
        file_path = data_dir / selected_file
        if not file_path.exists():
            raise FileNotFoundError(f"Datei nicht gefunden: {file_path}")
        file_paths = [file_path]

    dfs = []
    for path in file_paths:
        df = pd.read_csv(
            path,
            sep=r"\s+",
            skiprows=4,
            names=COLUMNS,
            encoding="latin1",
            engine="python",
        )

        parts = path.stem.split("_")
        material, temperature, mode = parts[:3] if len(parts) >= 3 else (path.stem, None, None)
        first_freq = df["Freq_Hz"].iloc[0]
        df["Spectrum_ID"] = (df["Freq_Hz"] == first_freq).cumsum() - 1
        df["Spectrum_Number"] = df["Spectrum_ID"] + 1
        df["Point_Number"] = range(1, len(df) + 1)
        df["Material"] = material
        df["Temperature"] = temperature
        df["Mode"] = mode
        df["Source_File"] = path.name
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def add_relative_switch_time(df, switch_points):
    df = df.copy()
    switch_rows = []

    for dataset_key, dataset in df.groupby(["Material", "Temperature", "Mode"], dropna=False):
        switch_point_number = switch_points.get(dataset_key)
        if switch_point_number is None:
            raise ValueError(f"Fuer diese Reihe fehlt eine Switch_Point_Number: {dataset_key}")

        switch_point = dataset[dataset["Point_Number"] == switch_point_number]
        next_point = dataset[dataset["Point_Number"] == switch_point_number + 1]
        if switch_point.empty or next_point.empty:
            raise ValueError(f"Switch-Point-Paar nicht gefunden: {dataset_key}, {switch_point_number}")

        switch_time = float((switch_point["MTime_s"].iloc[0] + next_point["MTime_s"].iloc[0]) / 2)
        switch_rows.append({
            "Material": dataset_key[0],
            "Temperature": dataset_key[1],
            "Mode": dataset_key[2],
            "Switch_Point_Number": int(switch_point_number),
            "Switch_Time_s": switch_time,
        })

    switch_times = pd.DataFrame(switch_rows)
    df = df.merge(switch_times, on=["Material", "Temperature", "Mode"], how="left")
    df[TIME_COL] = df["MTime_s"] - df["Switch_Time_s"]
    return df, switch_times


df_raw = load_measurements(selected_file=SELECTED_FILE)
df_raw, switch_times = add_relative_switch_time(df_raw, manual_switch_points)
switch_times


## Interpolation und Ableitung

In [ ]:
def _linear_at(target_time, times, values):
    if len(times) < 2:
        return np.nan
    x0, x1 = float(times[0]), float(times[1])
    y0, y1 = float(values[0]), float(values[1])
    if x0 == x1:
        return np.nan
    return y0 + (target_time - x0) * (y1 - y0) / (x1 - x0)


def _insert_switch_support_point(freq_data, value_col, time_col=TIME_COL, target_time=0):
    before = freq_data[freq_data[time_col] < target_time].tail(2)
    support_points = freq_data[[time_col, value_col]].dropna().copy()
    support_points = support_points[~np.isclose(support_points[time_col], target_time)]

    if len(before) >= 2:
        switch_value = _linear_at(target_time, before[time_col].to_numpy(), before[value_col].to_numpy())
        support_points = pd.concat(
            [support_points, pd.DataFrame({time_col: [float(target_time)], value_col: [switch_value]})],
            ignore_index=True,
        )

    support_points = support_points.sort_values(time_col)
    return support_points.groupby(time_col, as_index=False)[value_col].mean()


def interpolate_complete_spectra(
    df,
    time_col=TIME_COL,
    freq_col="Freq_Hz",
    value_cols=("Eps_real", "Eps_imag"),
    dataset_cols=("Source_File", "Material", "Temperature", "Mode"),
    drop_incomplete=True,
):
    rows = []

    for dataset_key, dataset in df.groupby(list(dataset_cols), dropna=False):
        dataset_meta = dict(zip(dataset_cols, dataset_key if isinstance(dataset_key, tuple) else (dataset_key,)))
        target_times = np.sort(dataset[time_col].dropna().unique())
        if not np.isclose(target_times, 0).any():
            target_times = np.sort(np.append(target_times, 0.0))
        frequencies = np.sort(dataset[freq_col].dropna().unique())

        interpolated_by_freq = {}
        for freq in frequencies:
            freq_data = dataset.loc[dataset[freq_col] == freq, [time_col, *value_cols]].dropna(subset=[time_col])
            freq_data = freq_data.sort_values(time_col).groupby(time_col, as_index=False)[list(value_cols)].mean()

            interpolated_values = {}
            for value_col in value_cols:
                support = _insert_switch_support_point(freq_data, value_col, time_col=time_col, target_time=0)
                interpolated_values[value_col] = np.interp(
                    target_times,
                    support[time_col].to_numpy(),
                    support[value_col].to_numpy(),
                    left=np.nan,
                    right=np.nan,
                )
            interpolated_by_freq[freq] = interpolated_values

        for spectrum_id, target_time in enumerate(target_times):
            for freq in frequencies:
                row = {
                    **dataset_meta,
                    "Interpolated_Spectrum_ID": spectrum_id,
                    time_col: target_time,
                    freq_col: freq,
                    "Is_Switch_Spectrum": bool(np.isclose(target_time, 0)),
                }
                for value_col in value_cols:
                    row[value_col] = interpolated_by_freq[freq][value_col][spectrum_id]
                rows.append(row)

    interpolated = pd.DataFrame(rows)

    if drop_incomplete:
        complete_ids = [*dataset_cols, "Interpolated_Spectrum_ID"]
        complete_mask = interpolated.groupby(complete_ids, dropna=False)[list(value_cols)].transform(
            lambda values: values.notna().all()
        )
        interpolated = interpolated[complete_mask.all(axis=1)].reset_index(drop=True)

    return interpolated


def derive_eps_real_by_frequency(
    df,
    value_col="Eps_real",
    freq_col="Freq_Hz",
    time_col=TIME_COL,
    ignored_last_points=IGNORE_LAST_FREQ_POINTS,
):
    rows = []
    group_cols = ["Source_File", "Material", "Temperature", "Mode", "Interpolated_Spectrum_ID", time_col]

    for group_key, spectrum in df.groupby(group_cols, dropna=False):
        spectrum = spectrum.sort_values(freq_col).reset_index(drop=True)
        if ignored_last_points:
            spectrum = spectrum.iloc[:-ignored_last_points]
        if len(spectrum) < 2:
            continue

        frequencies = spectrum[freq_col].to_numpy()
        eps_real = spectrum[value_col].to_numpy()
        omega = 2 * np.pi * frequencies
        ln_omega = np.log(omega)
        derivative = -np.pi / 2 * np.diff(eps_real) / np.diff(ln_omega)
        omega_mid = np.exp((ln_omega[:-1] + ln_omega[1:]) / 2)
        freq_mid = omega_mid / (2 * np.pi)
        meta = dict(zip(group_cols, group_key if isinstance(group_key, tuple) else (group_key,)))

        for freq, omega_value, derivative_value in zip(freq_mid, omega_mid, derivative):
            rows.append({
                **meta,
                "Freq_Hz_mid": freq,
                "Omega_rad_s": omega_value,
                "Eps_real_derivative": derivative_value,
            })

    return pd.DataFrame(rows)


df_interpolated = interpolate_complete_spectra(df_raw)
df_derivative = derive_eps_real_by_frequency(df_interpolated)
df_derivative.head()


## Cole-Cole-Modell

Standardmaessig werden hier zwei Cole-Cole-Terme verwendet. Das ist fuer den Zeitverlauf meist stabiler als drei Terme. Falls die Residuen systematisch schlecht bleiben oder ein Peak sichtbar fehlt, kann ein dritter Term wieder hinzugefuegt werden.

Achtung bei der Interpretation: Wenn ein `omega_p` sehr weit ausserhalb des gemessenen Frequenzfensters liegt, beschreibt dieser Term eher einen Rand-/Basisbeitrag als einen gut aufgeloesten Peak.


In [ ]:
TINY = 1e-30


def cc_imag(w, de, alpha, wp):
    den = 1 + (1j * w / wp) ** alpha
    return -np.imag(de / den)


def cc_real_derivative(w, de, alpha, wp):
    A = alpha * np.pi / 2
    W = (w / wp) ** alpha
    return (
        A
        * de
        * W
        * np.cos(A - 2 * np.arctan(np.sin(A) / (1 / W + np.cos(A))))
        / (1 + 2 * W * np.cos(A) + W**2)
    )


def _sum_terms(w, params, term_function):
    y = np.zeros_like(w, dtype=float)
    for de, alpha, wp in np.array(params).reshape(-1, 3):
        y += term_function(w, de, alpha, wp)
    return y


def derivative_log_model(lnw, *params):
    w = np.exp(lnw)
    return np.log(np.maximum(_sum_terms(w, params, cc_real_derivative), TINY))


def imag_log_model(lnw, *params):
    w = np.exp(lnw)
    return np.log(np.maximum(_sum_terms(w, params, cc_imag), TINY))


def combined_log_model(x_all, *params):
    lnw, mask = x_all
    return np.where(mask == 0, derivative_log_model(lnw, *params), imag_log_model(lnw, *params))


# Startwerte und Grenzen: bei Bedarf anpassen.
# Struktur je Term: Delta epsilon, alpha, omega_peak [rad/s]
# Zwei Terme sind der stabilere Startpunkt fuer Zeitverlaufs-Fits.
P0 = np.array([
    1.0, 0.7, 1e2,
    0.5, 0.5, 1e5,
], dtype=float)

LOWER_BOUNDS = np.array([
    1e-10, 0.05, 1e-8,
    1e-10, 0.05, 1e1,
], dtype=float)

UPPER_BOUNDS = np.array([
    np.inf, 1.0, 1e4,
    np.inf, 1.0, 1e9,
], dtype=float)

PARAMETER_LABELS = [
    "de_1", "alpha_1", "omega_p_1",
    "de_2", "alpha_2", "omega_p_2",
]

N_COLE_TERMS = len(P0) // 3


## Fit eines einzelnen Zeitpunktes

In [ ]:
def _clip_to_bounds(p0):
    return np.minimum(np.maximum(np.asarray(p0, dtype=float), LOWER_BOUNDS * 1.000001), UPPER_BOUNDS * 0.999999)


def _fit_quality(y_data, y_fit, popt):
    residual = y_data - y_fit
    rmse_log = float(np.sqrt(np.nanmean(residual**2)))
    ss_res = float(np.nansum(residual**2))
    ss_tot = float(np.nansum((y_data - np.nanmean(y_data)) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan
    lower_hit = np.isclose(popt, LOWER_BOUNDS, rtol=1e-4, atol=1e-12)
    upper_hit = np.isclose(popt, UPPER_BOUNDS, rtol=1e-4, atol=1e-12)
    at_bounds = bool(np.any(lower_hit | upper_hit))
    return {
        "RMSE_Log": rmse_log,
        "R2": r2,
        "At_Bounds": at_bounds,
        "Bounds_Hit_Count": int(np.sum(lower_hit | upper_hit)),
    }


def _is_fit_accepted(quality, max_rmse_log=1.0, min_r2=0.5, allow_bounds=False):
    if not np.isfinite(quality["RMSE_Log"]):
        return False
    if quality["RMSE_Log"] > max_rmse_log:
        return False
    if np.isfinite(quality["R2"]) and quality["R2"] < min_r2:
        return False
    if quality["At_Bounds"] and not allow_bounds:
        return False
    return True


def build_fit_data(fit_time_s, fit_eps_imag=FIT_EPS_IMAG):
    available_times = np.sort(df_derivative[TIME_COL].unique())
    plot_time = available_times[np.abs(available_times - fit_time_s).argmin()]

    der = df_derivative[df_derivative[TIME_COL] == plot_time].sort_values("Omega_rad_s")
    der = der[(der["Eps_real_derivative"] > 0) & np.isfinite(der["Eps_real_derivative"])]
    x_der = np.log(der["Omega_rad_s"].to_numpy())
    y_der = np.log(der["Eps_real_derivative"].to_numpy())

    if fit_eps_imag:
        imag = df_interpolated[df_interpolated[TIME_COL] == plot_time].sort_values("Freq_Hz")
        if IGNORE_LAST_FREQ_POINTS:
            imag = imag.iloc[:-IGNORE_LAST_FREQ_POINTS]
        imag = imag[(imag["Eps_imag"] > 0) & np.isfinite(imag["Eps_imag"])]
        x_imag = np.log(2 * np.pi * imag["Freq_Hz"].to_numpy())
        y_imag = np.log(imag["Eps_imag"].to_numpy())
        x_all = np.concatenate([x_der, x_imag])
        y_all = np.concatenate([y_der, y_imag])
        mask = np.concatenate([np.zeros_like(x_der), np.ones_like(x_imag)])
        model = combined_log_model
        x_fit = (x_all, mask)
    else:
        y_all = y_der
        model = derivative_log_model
        x_fit = x_der

    return plot_time, x_fit, y_all, model, der


def fit_spectrum_at_time(
    fit_time_s,
    p0=P0,
    fit_eps_imag=FIT_EPS_IMAG,
    max_rmse_log=1.0,
    min_r2=0.5,
    allow_bounds=False,
):
    plot_time, x_fit, y_data, model, der = build_fit_data(fit_time_s, fit_eps_imag=fit_eps_imag)
    p0 = _clip_to_bounds(p0)

    popt, pcov = curve_fit(
        model,
        x_fit,
        y_data,
        p0=p0,
        bounds=(LOWER_BOUNDS, UPPER_BOUNDS),
        maxfev=50000,
    )

    quality = _fit_quality(y_data, model(x_fit, *popt), popt)
    quality["Accepted"] = _is_fit_accepted(
        quality,
        max_rmse_log=max_rmse_log,
        min_r2=min_r2,
        allow_bounds=allow_bounds,
    )
    return plot_time, popt, pcov, der, quality


fit_time, popt, pcov, fit_derivative_df, fit_quality = fit_spectrum_at_time(FIT_TIME_S)
fit_result = pd.DataFrame({"parameter": PARAMETER_LABELS, "value": popt})
fit_result


In [ ]:
w_plot = np.logspace(
    np.log10(fit_derivative_df["Omega_rad_s"].min()),
    np.log10(fit_derivative_df["Omega_rad_s"].max()),
    400,
)

plt.figure(figsize=(9, 6))
plt.scatter(
    fit_derivative_df["Omega_rad_s"],
    fit_derivative_df["Eps_real_derivative"],
    color="black",
    label="Ableitung interpoliertes Spektrum",
)

fit_total = _sum_terms(w_plot, popt, cc_real_derivative)
plt.plot(w_plot, fit_total, color="red", linewidth=2.2, label="Cole-Cole-Fit gesamt")

for term_index, term_params in enumerate(np.array(popt).reshape(-1, 3), start=1):
    term_fit = cc_real_derivative(w_plot, *term_params)
    plt.plot(
        w_plot,
        term_fit,
        linestyle="--",
        linewidth=1.4,
        label=f"Cole-Cole-Term {term_index}",
    )

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Kreisfrequenz omega (rad/s)")
plt.ylabel("epsilon real derivative")
plt.title(f"Fit bei t_rel = {fit_time:.2f} s | {MATERIAL} {TEMPERATURE} {MODE}")
plt.legend()
plt.tight_layout()
plt.show()


## Zeitverlauf der Fitparameter

Der Zeitverlauf wird vom Referenzzeitpunkt aus in beide Richtungen gefittet. Das Ergebnis eines akzeptierten Fits wird als Startwert fuer den naechsten Zeitpunkt in derselben Richtung verwendet. Schlechte Fits bleiben in `df_fit_parameters` markiert, werden aber nicht als neuer Startwert weitergereicht.


In [ ]:
RUN_TIME_SERIES_FIT = False

FIT_TIME_MIN_S = -300
FIT_TIME_MAX_S = 900
FIT_EVERY_N = 5
FIT_REFERENCE_TIME_S = 0

# Fitqualitaetsgrenzen: bei Bedarf lockern oder strenger setzen.
MAX_RMSE_LOG = 1.0
MIN_R2 = 0.5
ALLOW_BOUND_HITS = False


def _fit_time_sequence(times, start_p0, direction_label):
    rows = []
    p0_current = start_p0.copy()

    for time_s in times:
        try:
            actual_time, popt_i, _, _, quality = fit_spectrum_at_time(
                time_s,
                p0=p0_current,
                max_rmse_log=MAX_RMSE_LOG,
                min_r2=MIN_R2,
                allow_bounds=ALLOW_BOUND_HITS,
            )
            row = {
                "Time_Relative_s": actual_time,
                "Fit_Success": True,
                "Direction": direction_label,
                **quality,
            }
            row.update(dict(zip(PARAMETER_LABELS, popt_i)))

            if quality["Accepted"]:
                p0_current = popt_i
        except Exception as err:
            row = {
                "Time_Relative_s": time_s,
                "Fit_Success": False,
                "Accepted": False,
                "Direction": direction_label,
                "Error": str(err),
            }
        rows.append(row)

    return rows


if RUN_TIME_SERIES_FIT:
    all_times = np.sort(df_derivative[TIME_COL].unique())
    all_times = all_times[(all_times >= FIT_TIME_MIN_S) & (all_times <= FIT_TIME_MAX_S)]
    all_times = all_times[::FIT_EVERY_N]

    if len(all_times) == 0:
        raise ValueError("Keine Zeitpunkte im gewaehlten Fit-Zeitfenster gefunden.")

    reference_time = all_times[np.abs(all_times - FIT_REFERENCE_TIME_S).argmin()]
    reference_time, reference_popt, _, _, reference_quality = fit_spectrum_at_time(
        reference_time,
        p0=P0,
        max_rmse_log=MAX_RMSE_LOG,
        min_r2=MIN_R2,
        allow_bounds=ALLOW_BOUND_HITS,
    )

    rows = []
    reference_row = {
        "Time_Relative_s": reference_time,
        "Fit_Success": True,
        "Direction": "reference",
        **reference_quality,
    }
    reference_row.update(dict(zip(PARAMETER_LABELS, reference_popt)))
    rows.append(reference_row)

    future_times = all_times[all_times > reference_time]
    past_times = all_times[all_times < reference_time][::-1]

    rows.extend(_fit_time_sequence(future_times, reference_popt, "forward"))
    rows.extend(_fit_time_sequence(past_times, reference_popt, "backward"))

    df_fit_parameters = (
        pd.DataFrame(rows)
        .sort_values("Time_Relative_s")
        .reset_index(drop=True)
    )

    print(df_fit_parameters.head())
else:
    df_fit_parameters = pd.DataFrame()
    print(
        "Zeitverlaufs-Fit ist deaktiviert. Setze RUN_TIME_SERIES_FIT = True, "
        "wenn die Parameter ueber alle ausgewaehlten Zeitpunkte gefittet werden sollen."
    )


In [ ]:
PARAMETERS_TO_PLOT = ["de_1", "de_2", "omega_p_1", "omega_p_2", "alpha_1", "alpha_2"]
PLOT_ONLY_ACCEPTED = True

if df_fit_parameters.empty:
    print(
        "Kein Zeitverlaufs-Fit vorhanden. Fuehre zuerst die vorherige Zelle mit "
        "RUN_TIME_SERIES_FIT = True aus."
    )
else:
    plot_params = df_fit_parameters[df_fit_parameters["Fit_Success"]].copy()
    if PLOT_ONLY_ACCEPTED:
        plot_params = plot_params[plot_params["Accepted"]]

    print(
        df_fit_parameters.groupby(["Fit_Success", "Accepted"], dropna=False)
        .size()
        .rename("count")
        .reset_index()
    )

    for parameter in PARAMETERS_TO_PLOT:
        plt.figure(figsize=(9, 5))
        plt.plot(plot_params["Time_Relative_s"], plot_params[parameter], marker="o")
        if parameter.startswith("omega") or parameter.startswith("de"):
            plt.yscale("log")
        plt.axvline(0, color="black", linestyle="--", linewidth=1)
        plt.xlabel("Zeit relativ zum Atmosphaerenwechsel (s)")
        plt.ylabel(parameter)
        plt.title(f"Zeitverlauf {parameter} | {MATERIAL} {TEMPERATURE} {MODE}")
        plt.tight_layout()
        plt.show()

    quality_cols = ["Time_Relative_s", "Direction", "Fit_Success", "Accepted", "RMSE_Log", "R2", "At_Bounds", "Bounds_Hit_Count"]
    print(df_fit_parameters[quality_cols].head(20))
